#### Processos suspeitos (detecção básica)

In [115]:
import psutil

def detectar_processos_suspeitos():
    suspeitos = []

    for proc in psutil.process_iter(['name', 'cpu_percent', 'memory_info']):
        try:
            cpu = proc.cpu_percent(interval=0.1)
            mem = proc.info['memory_info'].rss / (1024**3)
            nome = proc.info['name']

            if cpu > 80 or mem > 1:  # thresholds simples
                suspeitos.append({
                    "nome": nome,
                    "cpu_%": cpu,
                    "memoria_GB": round(mem, 2)
                })

        except:
            continue

    return suspeitos

detectar_processos_suspeitos()

[{'nome': 'System Idle Process', 'cpu_%': 752.8, 'memoria_GB': 0.0}]

#### Uso de memória por tipo

In [116]:
def memoria_detalhada():
    mem = psutil.virtual_memory()

    return {
        "total": round(mem.total / (1024**3), 2),
        "disponivel": round(mem.available / (1024**3), 2),
        "usada": round(mem.used / (1024**3), 2),
        "percentual": mem.percent
    }
    
memoria_detalhada()

{'total': 15.39, 'disponivel': 3.42, 'usada': 11.97, 'percentual': 77.8}

#### Tempo de boot da máquina/ Ótimo pra detectar máquina “travando ao longo do tempo”:

In [117]:
import datetime

def tempo_ligado():
    boot = datetime.datetime.fromtimestamp(psutil.boot_time())
    agora = datetime.datetime.now()

    return str(agora - boot)
tempo_ligado()

'5 days, 10:00:28.525963'

#### Detector de “máquina lenta”

In [118]:
def diagnostico_performance():
    cpu = psutil.cpu_percent(interval=1)
    mem = psutil.virtual_memory().percent

    if cpu > 85:
        return "CPU sobrecarregada"

    if mem > 90:
        return "Memória quase cheia"

    return "OK"
diagnostico_performance()

'OK'

#### Geolocalização do IP (nível externo)

In [120]:
import requests

def geo_ip():
    try:
        return requests.get("https://ipinfo.io/json").json()
    except:
        return "Erro ao obter localização"
    
geo_ip()

'Erro ao obter localização'

In [127]:
import psutil
import time
import socket
import statistics
import logging
import os
import json
import threading
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Optional

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# ================= CONFIG =================
CONFIG = {
    "interval": 2,
    "history_size": 60,
    "alert_cooldown": 10,
    "export_file": "monitor_log.json"
}

# ================= METRIC =================
@dataclass
class Metric:
    name: str
    history: List[float] = field(default_factory=list)
    max_history: int = CONFIG["history_size"]
    threshold_z: float = 2.5

    def add(self, value: float):
        if value is not None:
            self.history.append(value)
            if len(self.history) > self.max_history:
                self.history.pop(0)

    @property
    def avg(self):
        return statistics.mean(self.history) if self.history else 0

    @property
    def moving_avg(self):
        if len(self.history) < 5:
            return self.avg
        return statistics.mean(self.history[-5:])

    def is_anomalous(self, value: float):
        if len(self.history) < 10 or value is None:
            return False
        std = statistics.stdev(self.history)
        if std == 0:
            return False
        z = abs(value - self.avg) / std
        return z > self.threshold_z


# ================= MONITOR =================
class HealthMonitor:
    def __init__(self):
        self.metrics = {
            "cpu": Metric("CPU"),
            "mem": Metric("MEM"),
            "lat": Metric("LAT", threshold_z=3),
            "disk": Metric("DISK")
        }

        self.known_procs = {p.info['name'] for p in psutil.process_iter(['name'])}
        self.last_alert_time = 0
        self.data_log = []

    # -------- NETWORK --------
    def get_latency(self):
        try:
            start = time.perf_counter()
            with socket.create_connection(("8.8.8.8", 53), timeout=1.5):
                return (time.perf_counter() - start) * 1000
        except:
            return None

    # -------- COLLECT --------
    def collect(self):
        return {
            "cpu": psutil.cpu_percent(),
            "mem": psutil.virtual_memory().percent,
            "disk": psutil.disk_usage('/').percent,
            "lat": self.get_latency(),
            "time": datetime.now().isoformat()
        }

    # -------- PROCESS --------
    def analyze(self, data):
        alerts = []
        risk = 0

        for key, value in data.items():
            if key == "time":
                continue

            metric = self.metrics[key]

            if value is None:
                alerts.append(f"CRITICAL: {key} unreachable")
                risk += 40
                continue

            if metric.is_anomalous(value):
                alerts.append(f"ANOMALY: {key} spike ({value:.1f})")
                risk += 25

            metric.add(value)

        # processos novos
        current = {p.info['name'] for p in psutil.process_iter(['name'])}
        new = current - self.known_procs
        if new:
            alerts.append(f"NEW PROC: {list(new)[:3]}")
            risk += 10
            self.known_procs.update(current)

        return alerts, risk

    # -------- EXPORT --------

    def export(self, data):
        # garante que a pasta existe
        os.makedirs("data", exist_ok=True)

        self.data_log.append(data)
        if len(self.data_log) > 100:
            self.data_log.pop(0)

        file_path = os.path.join("data", CONFIG["export_file"])

        with open(file_path, "w") as f:
            json.dump(self.data_log, f, indent=2)

    # -------- REPORT --------
    def report(self, data, alerts, risk):
        os.system('cls' if os.name == 'nt' else 'clear')

        status = "🟢 OK" if risk < 30 else "🟡 WARN" if risk < 60 else "🔴 CRITICAL"

        print(f"=== MONITOR PRO === {datetime.now().strftime('%H:%M:%S')} | {status}")
        print(f"CPU  : {data['cpu']:.1f}% (M5: {self.metrics['cpu'].moving_avg:.1f})")
        print(f"MEM  : {data['mem']:.1f}%")
        print(f"DISK : {data['disk']:.1f}%")
        print(f"LAT  : {data['lat']:.1f}ms" if data['lat'] else "LAT  : TIMEOUT")
        print(f"RISK : {risk}/100")

        uptime = time.time() - psutil.boot_time()
        print(f"UPTIME: {int(uptime // 3600)}h")

        if alerts:
            print("\n[ALERTS]")
            for a in alerts:
                print(" -", a)

        # cooldown de alertas
        now = time.time()
        if risk > 50 and now - self.last_alert_time > CONFIG["alert_cooldown"]:
            logging.warning(f"ALERTA CRÍTICO: {alerts}")
            self.last_alert_time = now

    # -------- LOOP --------
    def run(self):
        logging.info("Monitor iniciado")

        while True:
            data = self.collect()

            # thread para export (não trava loop)
            threading.Thread(target=self.export, args=(data,), daemon=True).start()

            alerts, risk = self.analyze(data)
            self.report(data, alerts, risk)

            time.sleep(CONFIG["interval"])


if __name__ == "__main__":
    HealthMonitor().run()

2026-04-26 18:11:20,343 [INFO] Monitor iniciado


=== MONITOR PRO === 18:11:20 | 🟢 OK
CPU  : 4.0% (M5: 4.0)
MEM  : 76.1%
DISK : 82.5%
LAT  : 39.4ms
RISK : 0/100
UPTIME: 130h
=== MONITOR PRO === 18:11:22 | 🟢 OK
CPU  : 5.8% (M5: 4.9)
MEM  : 76.1%
DISK : 82.5%
LAT  : 30.7ms
RISK : 0/100
UPTIME: 130h
=== MONITOR PRO === 18:11:24 | 🟢 OK
CPU  : 5.1% (M5: 5.0)
MEM  : 76.1%
DISK : 82.5%
LAT  : 18.9ms
RISK : 0/100
UPTIME: 130h
=== MONITOR PRO === 18:11:26 | 🟢 OK
CPU  : 1.9% (M5: 4.2)
MEM  : 76.1%
DISK : 82.5%
LAT  : 42.7ms
RISK : 0/100
UPTIME: 130h
=== MONITOR PRO === 18:11:28 | 🟢 OK
CPU  : 10.3% (M5: 5.4)
MEM  : 76.3%
DISK : 82.5%
LAT  : 43.0ms
RISK : 0/100
UPTIME: 130h
=== MONITOR PRO === 18:11:30 | 🟢 OK
CPU  : 2.3% (M5: 5.1)
MEM  : 76.4%
DISK : 82.5%
LAT  : 19.3ms
RISK : 0/100
UPTIME: 130h
=== MONITOR PRO === 18:11:32 | 🟢 OK
CPU  : 1.4% (M5: 4.2)
MEM  : 76.2%
DISK : 82.5%
LAT  : 39.5ms
RISK : 0/100
UPTIME: 130h
=== MONITOR PRO === 18:11:35 | 🟢 OK
CPU  : 2.1% (M5: 3.6)
MEM  : 76.2%
DISK : 82.5%
LAT  : 36.9ms
RISK : 0/100
UPTIME: 130h
=== MON

KeyboardInterrupt: 